# Frame Captioning Extractor v1

Đọc keyframe từ GCS manifest, tạo caption bằng một trong ba model:

- `Qwen/Qwen2.5-VL-3B-Instruct`
- `Salesforce/blip2-opt-2.7b`
- `Qwen/Qwen3-VL-8B-Instruct`

Notebook giữ cùng cấu trúc chạy `dry / demo / full` và contract artifact JSONL như `fe-ocr-v1.ipynb`.  
Model được load tuần tự; benchmark không giữ nhiều model đồng thời trong VRAM.


## 1. Parameters

### Cách chạy trên Kaggle

1. Bật **GPU** và **Internet** trong Notebook Settings.
2. Chỉnh GCS, batch, model và batch size trong cell Parameters bên dưới.
3. Chạy lần lượt: **Parameters → Install Dependencies → Shared Helpers → Captioning Model Logic**.
4. Chạy **Dry Run** để kiểm tra credential, manifest và đường dẫn dự kiến. Dry Run không tải ảnh và không load model.
5. Chạy cell **Benchmark 5 videos** ngay dưới Dry Run để so sánh output, thời gian và VRAM của ba model.
6. Chạy **Demo Run** để kiểm tra end-to-end với `ACTIVE_MODEL_KEY`.
7. Chỉ chạy **Full Run** sau khi đặt `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"` và chạy lại cell Parameters.

### Chọn model

- Full/Demo dùng model trong `ACTIVE_MODEL_KEY`.
- Benchmark dùng danh sách `BENCHMARK_MODEL_KEYS`.
- Để thêm model cùng backend, chỉ cần thêm một entry vào `MODEL_REGISTRY`.
- Để thêm một kiến trúc hoàn toàn mới, thêm backend mới trong `load_caption_model()` và `generate_caption_batch()`.

### Gợi ý VRAM

`Qwen3-VL-8B-Instruct` mặc định dùng 4-bit để phù hợp GPU khoảng 16 GB. Đổi `quantization` thành `"none"` khi dùng GPU VRAM lớn và muốn so sánh cùng precision.


In [19]:
from kaggle_secrets import UserSecretsClient
secret_label_1 = "GCS_BUCKET"
secret_value = UserSecretsClient().get_secret(secret_label)
secret_value

'aic_ai_2026'

In [20]:
from types import SimpleNamespace

# ---------------------------------------------------------------------------
# GCS and dataset identity
# ---------------------------------------------------------------------------
GCS_BUCKET = "aic_ai_2026"
GCS_BUCKET_SECRET_NAME = "GCS_BUCKET"
GCS_CREDENTIALS_FILE = "/kaggle/input/datasets/zintom/secrets/gen-lang-client-0547522732-410672fac05f.json"
GCS_CREDENTIALS_JSON_SECRET_NAME = "GCS_CREDENTIALS_JSON"

DATASET_ID = "ai_challenge_2025"
PROFILE_VERSION = "autoshot_v1"
BATCHES = [
    "L21",
    # "L22",
    # "L23",
    # "L24",
    # "L26",
    # "L27",
    # "L28",
    # "L29",
    # "L30",
]

# Frame extraction layout on GCS.
KEYFRAMES_PREFIX = "processed/keyframes"
MANIFESTS_PREFIX = "processed/keyframes_manifests"
INPUT_MANIFEST_URI = ""  # Optional explicit gs://.../shot_segments.csv or frames_manifest.jsonl.

# ---------------------------------------------------------------------------
# Caption prompts
# ---------------------------------------------------------------------------
CAPTION_PROMPT_VI = """
Bạn là hệ thống tạo caption cho ảnh/keyframe video.

Nhiệm vụ:
- Mô tả nội dung chính của ảnh bằng tiếng Việt.
- Tập trung vào người, hành động, vật thể, bối cảnh, địa điểm và sự kiện.
- Nếu có chữ, logo, biển báo hoặc phụ đề quan trọng thì ghi lại phần nhìn thấy rõ.
- Không suy đoán thông tin không chắc chắn.
- Không mô tả quá dài.

Định dạng trả về:
Một đoạn văn tiếng Việt ngắn gọn, từ 1 đến 3 câu.
""".strip()

# Giữ prompt gốc của file get-features cho BLIP-2.
BLIP_PROMPT = "a photo of"

# ---------------------------------------------------------------------------
# Model registry
# Add another model with an existing backend by copying one entry below.
# Supported backends: qwen25_vl, blip2, qwen3_vl
# ---------------------------------------------------------------------------
MODEL_REGISTRY = {
    "qwen25_vl_3b": {
        "backend": "qwen25_vl",
        "hf_id": "Qwen/Qwen2.5-VL-3B-Instruct",
        "prompt": CAPTION_PROMPT_VI,
        "batch_size": 4,
        "max_new_tokens": 80,
        "quantization": "none",
        "max_pixels": 1024 * 1024,
    },
    "blip2_opt_2_7b": {
        "backend": "blip2",
        "hf_id": "Salesforce/blip2-opt-2.7b",
        "prompt": BLIP_PROMPT,
        "batch_size": 8,
        "max_new_tokens": 80,
        "min_new_tokens": 10,
        "num_beams": 1,
        "repetition_penalty": 1.2,
        "length_penalty": 1.1,
        "quantization": "none",
    },
    "qwen3_vl_8b": {
        "backend": "qwen3_vl",
        "hf_id": "Qwen/Qwen3-VL-8B-Instruct",
        "prompt": CAPTION_PROMPT_VI,
        "batch_size": 1,
        "max_new_tokens": 96,
        "quantization": "4bit",  # Set "none" on a large-VRAM GPU.
        "max_pixels": 1024 * 1024,
    },
}

ACTIVE_MODEL_KEY = "qwen25_vl_3b"
BENCHMARK_MODEL_KEYS = [
    "qwen25_vl_3b",
    "blip2_opt_2_7b",
    "qwen3_vl_8b",
]

# ---------------------------------------------------------------------------
# Extractor output layout on GCS
# The model key is included so resume does not mix annotations from models.
# ---------------------------------------------------------------------------
OUTPUT_PREFIX = "features/extractors"
EXTRACTOR_VERSION = f"fe-captioning-v1-{ACTIVE_MODEL_KEY}"
ANNOTATION_VERSION = "fe-captioning-v1"
MODEL_VERSION = MODEL_REGISTRY[ACTIVE_MODEL_KEY]["hf_id"]

# Kaggle local runtime.
RUN_ROOT = "/kaggle/working/feature_extractor_runs"
SCRATCH_DIR = "/kaggle/working/feature_extractor_scratch"
HF_CACHE_DIR = "/kaggle/working/huggingface_cache"
CLEANUP_LOCAL_FRAMES_AFTER_RUN = True

# ---------------------------------------------------------------------------
# Execution controls
# ---------------------------------------------------------------------------
DRY_RUN_MAX_FRAMES = 20

# Benchmark selects distinct video_id values and representative keyframes.
BENCHMARK_BATCHES = ["L21"]
BENCHMARK_VIDEO_LIMIT = 5
BENCHMARK_FRAMES_PER_VIDEO = 1
BENCHMARK_SAVE_CSV = True

DEMO_BATCHES = ["L21"]
DEMO_MAX_FRAMES = 32
FULL_MAX_FRAMES = None
CONFIRM_FULL_RUN = ""  # Set to RUN_FULL_DATASET before full run.

# Resume and failure behavior.
UPLOAD_TO_GCS = True
UPLOAD_RUN_ARTIFACTS = True
SKIP_EXISTING = True
OVERWRITE = False
RESUME_ANNOTATIONS_URI = ""
FAIL_FAST = False

# Parallelism and progress.
DOWNLOAD_WORKERS = 8
UPLOAD_WORKERS = 8
PIPELINE_BATCH_SIZE = 8
LOG_EVERY_N_FRAMES = 32
USE_TQDM = True

# Model runtime.
DEVICE = "auto"  # "auto", "cuda", or "cpu"
USE_FLASH_ATTENTION_2 = False
TRUST_REMOTE_CODE = False
LOW_CPU_MEM_USAGE = True

cfg = SimpleNamespace(**{
    name: value
    for name, value in globals().copy().items()
    if name.isupper() and not name.startswith("_")
})
cfg.EXTRACTOR_NAME = "captioning"

if cfg.ACTIVE_MODEL_KEY not in cfg.MODEL_REGISTRY:
    raise KeyError(f"Unknown ACTIVE_MODEL_KEY={cfg.ACTIVE_MODEL_KEY}")
unknown_benchmark_models = [
    key for key in cfg.BENCHMARK_MODEL_KEYS
    if key not in cfg.MODEL_REGISTRY
]
if unknown_benchmark_models:
    raise KeyError(f"Unknown BENCHMARK_MODEL_KEYS={unknown_benchmark_models}")

print("Parameters loaded for Frame Captioning Extractor v1.")
print("Active model:", cfg.ACTIVE_MODEL_KEY, cfg.MODEL_REGISTRY[cfg.ACTIVE_MODEL_KEY]["hf_id"])
print("Benchmark models:", cfg.BENCHMARK_MODEL_KEYS)
print("Batches:", cfg.BATCHES, "Demo:", cfg.DEMO_BATCHES, "Upload:", cfg.UPLOAD_TO_GCS)


Parameters loaded for Frame Captioning Extractor v1.
Active model: qwen25_vl_3b Qwen/Qwen2.5-VL-3B-Instruct
Benchmark models: ['qwen25_vl_3b', 'blip2_opt_2_7b', 'qwen3_vl_8b']
Batches: ['L21'] Demo: ['L21'] Upload: True


## 2. Install Dependencies

**Note:** Chạy cell này một lần sau khi mở Kaggle session. Nếu package được nâng cấp trong session đang chạy, restart kernel rồi chạy lại từ cell Parameters.


In [5]:
%pip install -q -U \
    "transformers>=4.57.0,<5" \
    accelerate \
    qwen-vl-utils \
    bitsandbytes \
    sentencepiece \
    google-cloud-storage \
    pandas \
    tqdm \
    numpy \
    pillow

# Optional, only when the Kaggle GPU/runtime supports it:
# %pip install -q flash-attn --no-build-isolation


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.4/341.4 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 102.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.7/676.7 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 84.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 105.2 MB/s eta 0:00:0000:01


## 3. Shared GCS, Manifest, Run Helpers

**Note:** Cell này giữ helper chung từ format `fe-ocr-v1.ipynb`: đọc manifest, tải frame, resume, ghi artifact và upload lên GCS.


In [21]:
from __future__ import annotations

import csv
import json
import logging
import os
import shutil
import time
import uuid
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any, Iterable

import numpy as np
import pandas as pd
from tqdm.auto import tqdm


@dataclass
class RunLayout:
    """Local and GCS paths for one extractor run."""
    run_id: str
    run_dir: Path
    frames_dir: Path
    artifacts_dir: Path
    output_prefix: str
    annotations_path: Path
    errors_path: Path
    metrics_path: Path
    summary_path: Path
    log_path: Path


def utc_now() -> str:
    """Return an ISO-8601 UTC timestamp."""
    return datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")


def cfg_value(config: Any, name: str, default: Any = None) -> Any:
    """Read a value from a SimpleNamespace-like config object."""
    return getattr(config, name, default)


def normalize_prefix(value: str) -> str:
    """Normalize a GCS object prefix without leading/trailing slashes."""
    return str(value or "").strip().strip("/")


def make_run_id(kind: str) -> str:
    """Create a unique, sortable run identifier."""
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{kind}_{stamp}_{uuid.uuid4().hex[:8]}"


def read_kaggle_secret(secret_name: str) -> str:
    """Read a Kaggle secret if the notebook is running on Kaggle."""
    if not secret_name:
        return ""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name) or ""
    except Exception:
        return ""


def resolve_bucket_name(config: Any, require: bool = True) -> str:
    """Resolve the target GCS bucket from params, env vars, or Kaggle Secrets."""
    configured = str(cfg_value(config, "GCS_BUCKET", "") or "").strip()
    env_value = os.environ.get("GCS_BUCKET", "").strip()
    secret_name = str(cfg_value(config, "GCS_BUCKET_SECRET_NAME", "GCS_BUCKET") or "").strip()
    resolved = configured or env_value or read_kaggle_secret(secret_name).strip()
    if resolved.startswith("gs://"):
        resolved = resolved[len("gs://"):].split("/", 1)[0]
    if require and not resolved:
        raise RuntimeError("Set GCS_BUCKET in params, env vars, or Kaggle Secrets.")
    return resolved


def make_storage_client(config: Any):
    """Create a google-cloud-storage client using Kaggle Secrets or ADC."""
    from google.cloud import storage

    credentials_file = str(cfg_value(config, "GCS_CREDENTIALS_FILE", "") or os.environ.get("GCS_CREDENTIALS_FILE", "")).strip()
    credentials_json = os.environ.get("GCS_CREDENTIALS_JSON", "").strip()
    secret_name = str(cfg_value(config, "GCS_CREDENTIALS_JSON_SECRET_NAME", "GCS_CREDENTIALS_JSON") or "").strip()
    credentials_json = credentials_json or read_kaggle_secret(secret_name).strip()
    if credentials_json:
        from google.oauth2 import service_account
        credentials = service_account.Credentials.from_service_account_info(json.loads(credentials_json))
        return storage.Client(project=credentials.project_id, credentials=credentials)
    if credentials_file:
        return storage.Client.from_service_account_json(credentials_file)
    return storage.Client()


def parse_gcs_uri(uri: str) -> tuple[str, str]:
    """Parse gs://bucket/object into bucket and object name."""
    if not str(uri).startswith("gs://"):
        raise ValueError(f"Expected gs:// URI, got {uri}")
    bucket, _, blob = str(uri)[5:].partition("/")
    if not bucket or not blob:
        raise ValueError(f"Invalid GCS URI: {uri}")
    return bucket, blob


def make_output_prefix(config: Any, batch_id: str, run_id: str) -> str:
    """Build the task output prefix for one logical batch/run."""
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/"
        f"frame_profile={cfg_value(config, 'PROFILE_VERSION')}/"
        f"extractor={cfg_value(config, 'EXTRACTOR_NAME')}/"
        f"extractor_version={cfg_value(config, 'EXTRACTOR_VERSION')}/"
        f"run_id={run_id}/"
    )


def make_run_layout(config: Any, batch_id: str, run_kind: str) -> RunLayout:
    """Create local directories and output files for a run."""
    run_id = make_run_id(run_kind)
    run_dir = Path(str(cfg_value(config, "RUN_ROOT", "/kaggle/working/feature_extractor_runs"))) / run_id
    frames_dir = run_dir / "frames"
    artifacts_dir = run_dir / "artifacts"
    frames_dir.mkdir(parents=True, exist_ok=True)
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    return RunLayout(run_id, run_dir, frames_dir, artifacts_dir, make_output_prefix(config, batch_id, run_id), artifacts_dir / "annotations.jsonl", artifacts_dir / "errors.jsonl", artifacts_dir / "metrics.csv", artifacts_dir / "summary.json", run_dir / "run.log")


def setup_logging(layout: RunLayout, verbose: bool = False) -> logging.Logger:
    """Configure console and file logging for a notebook run."""
    logger = logging.getLogger(str(cfg_value(cfg, "EXTRACTOR_NAME", "feature_extractor")))
    logger.setLevel(logging.DEBUG if verbose else logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s %(levelname)s %(message)s")
    stream = logging.StreamHandler()
    stream.setFormatter(fmt)
    file_handler = logging.FileHandler(layout.log_path, encoding="utf-8")
    file_handler.setFormatter(fmt)
    logger.addHandler(stream)
    logger.addHandler(file_handler)
    return logger


def upload_file(bucket: Any, local_path: Path, object_key: str, content_type: str = "application/octet-stream") -> None:
    """Upload one local file to GCS."""
    bucket.blob(object_key).upload_from_filename(str(local_path), content_type=content_type, timeout=900)


def upload_text(bucket: Any, text: str, object_key: str, content_type: str = "text/plain") -> None:
    """Upload text content to GCS."""
    bucket.blob(object_key).upload_from_string(text, content_type=content_type, timeout=300)


def write_json(path: Path, payload: dict[str, Any]) -> None:
    """Write a JSON object to disk with UTF-8 encoding."""
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def append_jsonl(path: Path, records: Iterable[dict[str, Any]]) -> int:
    """Append JSONL records to a local file and return the row count."""
    count = 0
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1
    return count


def append_metric(path: Path, row: dict[str, Any]) -> None:
    """Append one row to metrics.csv, creating the header if needed."""
    exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def latest_blob_name(bucket: Any, prefix: str, suffix: str) -> str | None:
    """Return the latest blob under prefix with the requested suffix."""
    blobs = [blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith(suffix)]
    if not blobs:
        return None
    blobs.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return blobs[0].name


def find_manifest_blobs(config: Any, bucket: Any, batches: list[str]) -> list[str]:
    """Find GCS manifest files for selected batches."""
    explicit = str(cfg_value(config, "INPUT_MANIFEST_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"INPUT_MANIFEST_URI bucket {parsed_bucket} does not match {bucket.name}")
        return [blob_name]
    blobs: list[str] = []
    base = normalize_prefix(cfg_value(config, "MANIFESTS_PREFIX", "processed/keyframes_manifests"))
    for batch_id in batches:
        prefix = f"{base}/dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/profile={cfg_value(config, 'PROFILE_VERSION')}/"
        name = latest_blob_name(bucket, prefix, "shot_segments.csv") or latest_blob_name(bucket, prefix, "frames_manifest.jsonl")
        if name is None:
            raise FileNotFoundError(f"No shot_segments.csv or frames_manifest.jsonl found under gs://{bucket.name}/{prefix}")
        blobs.append(name)
    return blobs


def read_manifest_blob(bucket: Any, blob_name: str) -> pd.DataFrame:
    """Read a CSV or JSONL frame manifest from GCS into a DataFrame."""
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    if blob_name.endswith(".csv"):
        return pd.read_csv(StringIO(text))
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    return pd.DataFrame(rows)


def normalize_manifest_records(df: pd.DataFrame, config: Any, batch_id: str | None = None) -> list[dict[str, Any]]:
    """Normalize frame manifest columns into the extractor record contract."""
    if df.empty:
        return []
    df = df.copy()
    if "saved" in df.columns:
        df = df[df["saved"].astype(str).str.lower().isin(["true", "1", "yes"])]
    records: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        image_gcs_uri = str(row.get("image_gcs_uri") or row.get("gcs_uri") or row.get("image_uri") or "").strip()
        if not image_gcs_uri:
            continue
        video_id = str(row.get("video_id") or Path(image_gcs_uri).parent.name).strip()
        frame_idx = int(float(row.get("frame_idx", 0) or 0))
        keyframe_id = str(row.get("keyframe_id") or f"{video_id}_F{frame_idx:06d}")
        records.append({
            "dataset_id": str(row.get("dataset_id") or cfg_value(config, "DATASET_ID")),
            "batch_id": str(row.get("batch_id") or batch_id or "").strip(),
            "video_id": video_id,
            "video_name": str(row.get("video_name") or ""),
            "shot_id": str(row.get("shot_id") or ""),
            "shot_start_frame": int(float(row.get("shot_start_frame", 0) or 0)),
            "shot_end_frame": int(float(row.get("shot_end_frame", 0) or 0)),
            "frame_type": str(row.get("frame_type") or ""),
            "frame_idx": frame_idx,
            "frame_sec": float(row.get("frame_sec", row.get("timestamp", 0)) or 0),
            "timestamp_ms": int(float(row.get("frame_sec", 0) or 0) * 1000),
            "keyframe_id": keyframe_id,
            "image_rel_path": str(row.get("image_rel_path") or f"{video_id}/{Path(image_gcs_uri).name}"),
            "image_gcs_uri": image_gcs_uri,
            "image_storage_key": str(row.get("image_storage_key") or parse_gcs_uri(image_gcs_uri)[1]),
            "fps": float(row.get("fps", 0) or 0),
            "profile_version": str(row.get("profile_version") or cfg_value(config, "PROFILE_VERSION")),
        })
    return records


def discover_frame_records(config: Any, bucket: Any, batches: list[str], max_frames: int | None = None) -> list[dict[str, Any]]:
    """Discover and normalize frame records for selected logical batches."""
    all_records: list[dict[str, Any]] = []
    for blob_name in find_manifest_blobs(config, bucket, batches):
        batch_hint = next((part.split("=", 1)[1] for part in blob_name.split("/") if part.startswith("batch=")), None)
        all_records.extend(normalize_manifest_records(read_manifest_blob(bucket, blob_name), config, batch_hint))
    all_records.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return all_records[: int(max_frames)] if max_frames is not None else all_records


def local_frame_path(layout: RunLayout, record: dict[str, Any]) -> Path:
    """Return the local scratch path for one frame record."""
    return layout.frames_dir / record["video_id"] / Path(record["image_gcs_uri"]).name


def download_one_frame(record: dict[str, Any], layout: RunLayout, client: Any) -> dict[str, Any]:
    """Download one frame from GCS into local scratch and return an updated record."""
    started = time.perf_counter()
    bucket_name, blob_name = parse_gcs_uri(record["image_gcs_uri"])
    out_path = local_frame_path(layout, record)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    if not out_path.exists() or out_path.stat().st_size == 0:
        client.bucket(bucket_name).blob(blob_name).download_to_filename(str(out_path), timeout=900)
    updated = dict(record)
    updated["local_image_path"] = str(out_path)
    updated["download_ms"] = int((time.perf_counter() - started) * 1000)
    return updated


def download_frames(records: list[dict[str, Any]], layout: RunLayout, client: Any, config: Any) -> list[dict[str, Any]]:
    """Download many frames concurrently from GCS."""
    downloaded: list[dict[str, Any]] = []
    workers = max(1, int(cfg_value(config, "DOWNLOAD_WORKERS", 8)))
    with ThreadPoolExecutor(max_workers=workers, thread_name_prefix="gcs-download") as pool:
        futures = [pool.submit(download_one_frame, record, layout, client) for record in records]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading frames", disable=not cfg_value(config, "USE_TQDM", True)):
            downloaded.append(future.result())
    downloaded.sort(key=lambda r: (r["batch_id"], r["video_id"], r["frame_idx"], r["keyframe_id"]))
    return downloaded


def iter_batches(items: list[Any], batch_size: int) -> Iterable[list[Any]]:
    """Yield fixed-size batches from a list."""
    batch_size = max(1, int(batch_size))
    for start in range(0, len(items), batch_size):
        yield items[start:start + batch_size]


def base_annotation(record: dict[str, Any], config: Any, kind: str, run_id: str) -> dict[str, Any]:
    """Create the shared frame annotation JSON payload."""
    return {
        "dataset_id": record["dataset_id"], "batch_id": record["batch_id"], "video_id": record["video_id"],
        "keyframe_id": record["keyframe_id"], "frame_id": record["keyframe_id"], "shot_id": record.get("shot_id", ""),
        "frame_idx": record["frame_idx"], "frame_sec": record["frame_sec"], "timestamp_ms": record.get("timestamp_ms", int(float(record.get("frame_sec", 0)) * 1000)),
        "frame_type": record.get("frame_type", ""), "image_gcs_uri": record["image_gcs_uri"], "image_storage_key": record.get("image_storage_key", ""),
        "kind": kind, "caption": None, "ocr_texts": [], "detected_objects": [], "object_counts": {}, "detections": [],
        "text_value": None, "json_value": {}, "confidence": 1.0, "model_version": str(cfg_value(config, "MODEL_VERSION", "unknown")),
        "annotation_version": str(cfg_value(config, "ANNOTATION_VERSION", cfg_value(config, "EXTRACTOR_VERSION", "v1"))), "run_id": run_id, "created_at": utc_now(),
    }


def upload_standard_artifacts(bucket: Any, layout: RunLayout, success: bool) -> None:
    """Upload standard run artifacts and optional _SUCCESS marker to GCS."""
    for path, content_type in [(layout.annotations_path, "application/jsonl"), (layout.errors_path, "application/jsonl"), (layout.metrics_path, "text/csv"), (layout.summary_path, "application/json"), (layout.log_path, "text/plain")]:
        if path.exists():
            upload_file(bucket, path, layout.output_prefix + path.name, content_type)
    if success:
        upload_text(bucket, "", layout.output_prefix + "_SUCCESS", "text/plain")



def output_base_prefix(config: Any, batch_id: str) -> str:
    """Build the extractor output prefix without run_id for resume discovery."""
    return (
        f"{normalize_prefix(cfg_value(config, 'OUTPUT_PREFIX', 'features/extractors'))}/"
        f"dataset={cfg_value(config, 'DATASET_ID')}/batch={batch_id}/"
        f"frame_profile={cfg_value(config, 'PROFILE_VERSION')}/"
        f"extractor={cfg_value(config, 'EXTRACTOR_NAME')}/"
        f"extractor_version={cfg_value(config, 'EXTRACTOR_VERSION')}/"
    )


def find_resume_annotations_blob(config: Any, bucket: Any, batches: list[str]) -> str | None:
    """Find an annotations.jsonl blob to use for resume filtering."""
    explicit = str(cfg_value(config, "RESUME_ANNOTATIONS_URI", "") or "").strip()
    if explicit:
        parsed_bucket, blob_name = parse_gcs_uri(explicit)
        if parsed_bucket != bucket.name:
            raise ValueError(f"RESUME_ANNOTATIONS_URI bucket {parsed_bucket} does not match {bucket.name}")
        return blob_name
    candidates = []
    for batch_id in batches:
        prefix = output_base_prefix(config, batch_id)
        candidates.extend([blob for blob in bucket.list_blobs(prefix=prefix) if blob.name.endswith("annotations.jsonl")])
    if not candidates:
        return None
    candidates.sort(key=lambda b: (b.updated or datetime.min.replace(tzinfo=timezone.utc), b.name), reverse=True)
    return candidates[0].name


def load_processed_keyframes(config: Any, bucket: Any, batches: list[str]) -> set[str]:
    """Load keyframe IDs already present in a previous successful annotations JSONL."""
    if not cfg_value(config, "SKIP_EXISTING", True) or cfg_value(config, "OVERWRITE", False):
        return set()
    blob_name = find_resume_annotations_blob(config, bucket, batches)
    if not blob_name:
        return set()
    text = bucket.blob(blob_name).download_as_text(timeout=900)
    processed: set[str] = set()
    for line in text.splitlines():
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except Exception:
            continue
        if not row.get("error") and row.get("keyframe_id"):
            processed.add(str(row["keyframe_id"]))
    return processed

def dry_run(config: Any, max_frames: int | None = None) -> dict[str, Any]:
    """Discover frame records without downloading images or running models."""
    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batches = [str(batch).upper() for batch in cfg_value(config, "BATCHES", [])]
    records = discover_frame_records(config, bucket, batches, max_frames=max_frames)
    return {"status": "DRY_RUN_OK", "bucket": bucket.name, "batches": batches, "planned_frames": len(records), "sample_records": records[:5]}


## 3b. Captioning Model Registry And Extraction Logic

**Note:** Full/Demo chỉ load `ACTIVE_MODEL_KEY`. Benchmark load từng model tuần tự, đo thời gian và VRAM, sau đó giải phóng model trước khi chuyển sang model tiếp theo.


In [22]:
import gc
from pathlib import Path
from typing import Any

def get_model_spec(config: Any, model_key: str) -> dict[str, Any]:
    """Return a defensive copy of one registry entry."""
    registry = cfg_value(config, "MODEL_REGISTRY", {})
    if model_key not in registry:
        raise KeyError(f"Model key {model_key!r} is not in MODEL_REGISTRY.")
    spec = dict(registry[model_key])
    spec["model_key"] = model_key
    return spec


def resolve_torch_device(config: Any) -> str:
    """Resolve the requested model device."""
    import torch

    requested = str(cfg_value(config, "DEVICE", "auto")).lower()
    if requested == "cpu":
        return "cpu"
    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("DEVICE='cuda' but CUDA is unavailable.")
        return "cuda"
    return "cuda" if torch.cuda.is_available() else "cpu"


def resolve_torch_dtype(device: str):
    """Use BF16 when supported, otherwise FP16 on CUDA and FP32 on CPU."""
    import torch

    if device == "cpu":
        return torch.float32
    if torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


def _quantization_config(spec: dict[str, Any], device: str):
    """Build a BitsAndBytesConfig or return None."""
    import torch
    from transformers import BitsAndBytesConfig

    mode = str(spec.get("quantization", "none")).lower()
    if mode == "none":
        return None
    if device != "cuda":
        raise RuntimeError(f"quantization={mode!r} requires CUDA.")
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    if mode == "4bit":
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
    if mode == "8bit":
        return BitsAndBytesConfig(load_in_8bit=True)
    raise ValueError(f"Unsupported quantization mode: {mode}")


def _first_model_device(model: Any):
    """Find the device used for input tensors."""
    try:
        return model.device
    except Exception:
        return next(model.parameters()).device


def reset_cuda_peak_memory() -> None:
    """Reset CUDA peak-memory counters on every visible GPU."""
    import torch

    if not torch.cuda.is_available():
        return
    torch.cuda.synchronize()
    for index in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(index)


def cuda_memory_metrics() -> dict[str, float]:
    """Return aggregate current and peak CUDA memory in GiB."""
    import torch

    if not torch.cuda.is_available():
        return {
            "vram_allocated_gb": 0.0,
            "vram_reserved_gb": 0.0,
            "peak_vram_allocated_gb": 0.0,
            "peak_vram_reserved_gb": 0.0,
        }

    torch.cuda.synchronize()
    scale = 1024 ** 3
    return {
        "vram_allocated_gb": round(sum(torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())) / scale, 4),
        "vram_reserved_gb": round(sum(torch.cuda.memory_reserved(i) for i in range(torch.cuda.device_count())) / scale, 4),
        "peak_vram_allocated_gb": round(sum(torch.cuda.max_memory_allocated(i) for i in range(torch.cuda.device_count())) / scale, 4),
        "peak_vram_reserved_gb": round(sum(torch.cuda.max_memory_reserved(i) for i in range(torch.cuda.device_count())) / scale, 4),
    }


def load_caption_model(
    config: Any,
    logger: logging.Logger,
    model_key: str | None = None,
) -> dict[str, Any]:
    """Load one caption model selected from MODEL_REGISTRY."""
    import torch
    from transformers import AutoProcessor

    model_key = model_key or str(cfg_value(config, "ACTIVE_MODEL_KEY"))
    spec = get_model_spec(config, model_key)
    backend = str(spec["backend"])
    model_id = str(spec["hf_id"])
    device = resolve_torch_device(config)
    dtype = resolve_torch_dtype(device)
    quantization_config = _quantization_config(spec, device)

    common_kwargs: dict[str, Any] = {
        "cache_dir": str(cfg_value(config, "HF_CACHE_DIR", "")) or None,
        "low_cpu_mem_usage": bool(cfg_value(config, "LOW_CPU_MEM_USAGE", True)),
        "trust_remote_code": bool(cfg_value(config, "TRUST_REMOTE_CODE", False)),
    }
    common_kwargs = {key: value for key, value in common_kwargs.items() if value is not None}

    if device == "cuda":
        common_kwargs["device_map"] = "auto"
        common_kwargs["torch_dtype"] = dtype
    else:
        common_kwargs["torch_dtype"] = torch.float32

    if quantization_config is not None:
        common_kwargs["quantization_config"] = quantization_config

    if bool(cfg_value(config, "USE_FLASH_ATTENTION_2", False)) and device == "cuda":
        common_kwargs["attn_implementation"] = "flash_attention_2"

    logger.info(
        "Loading model_key=%s backend=%s model_id=%s device=%s dtype=%s quantization=%s",
        model_key,
        backend,
        model_id,
        device,
        dtype,
        spec.get("quantization", "none"),
    )

    if backend == "qwen25_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(model_id, **common_kwargs)
    elif backend == "qwen3_vl":
        from transformers import Qwen3VLForConditionalGeneration
        model = Qwen3VLForConditionalGeneration.from_pretrained(model_id, **common_kwargs)
    elif backend == "blip2":
        from transformers import Blip2ForConditionalGeneration
        model = Blip2ForConditionalGeneration.from_pretrained(model_id, **common_kwargs)
        if device == "cpu":
            model = model.to("cpu")
    else:
        raise ValueError(
            f"Unsupported backend={backend!r}. "
            "Add a loader branch in load_caption_model() and an inference branch "
            "in generate_caption_batch()."
        )

    model.eval()
    processor = AutoProcessor.from_pretrained(
        model_id,
        cache_dir=str(cfg_value(config, "HF_CACHE_DIR", "")) or None,
        trust_remote_code=bool(cfg_value(config, "TRUST_REMOTE_CODE", False)),
    )
    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        processor.tokenizer.padding_side = "left"

    config.MODEL_VERSION = model_id
    return {
        "model_key": model_key,
        "spec": spec,
        "backend": backend,
        "model_id": model_id,
        "model": model,
        "processor": processor,
        "device": device,
        "dtype": dtype,
        "input_device": _first_model_device(model),
    }


def unload_caption_model(ctx: dict[str, Any] | None) -> None:
    """Delete model objects and release cached CUDA memory."""
    import torch

    if ctx:
        ctx.pop("model", None)
        ctx.pop("processor", None)
        ctx.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        gc.collect()


def _chunked(items: list[Any], batch_size: int):
    """Yield fixed-size model batches."""
    for start in range(0, len(items), max(1, int(batch_size))):
        yield items[start:start + max(1, int(batch_size))]


def _image_uri(image_path: str | Path) -> str:
    """Convert a local image path to the file URI expected by qwen-vl-utils."""
    return Path(image_path).expanduser().resolve().as_uri()


def _build_qwen_messages(
    image_path: str | Path,
    spec: dict[str, Any],
) -> list[dict[str, Any]]:
    """Build one Qwen chat conversation."""
    image_content: dict[str, Any] = {
        "type": "image",
        "image": _image_uri(image_path),
    }
    max_pixels = spec.get("max_pixels")
    if max_pixels:
        image_content["max_pixels"] = int(max_pixels)
    return [
        {
            "role": "user",
            "content": [
                image_content,
                {"type": "text", "text": str(spec["prompt"])},
            ],
        }
    ]


def _move_batch_to_device(inputs: Any, device: Any, dtype: Any | None = None):
    """Move a BatchFeature/dict while preserving integer tensor dtypes."""
    import torch

    moved = {}
    for key, value in inputs.items():
        if torch.is_tensor(value):
            if dtype is not None and torch.is_floating_point(value):
                moved[key] = value.to(device=device, dtype=dtype)
            else:
                moved[key] = value.to(device=device)
        else:
            moved[key] = value
    return moved


def _generate_qwen_batch(
    image_paths: list[str],
    ctx: dict[str, Any],
) -> list[str]:
    """Generate captions for Qwen2.5-VL or Qwen3-VL."""
    import torch
    from qwen_vl_utils import process_vision_info

    model = ctx["model"]
    processor = ctx["processor"]
    spec = ctx["spec"]
    backend = ctx["backend"]

    conversations = [_build_qwen_messages(path, spec) for path in image_paths]
    texts = [
        processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in conversations
    ]

    image_inputs: list[Any] = []
    video_inputs: list[Any] = []
    for messages in conversations:
        if backend == "qwen3_vl":
            patch_size = getattr(getattr(processor, "image_processor", None), "patch_size", 16)
            sample_images, sample_videos = process_vision_info(
                messages,
                image_patch_size=patch_size,
            )
        else:
            sample_images, sample_videos = process_vision_info(messages)

        if sample_images:
            image_inputs.extend(sample_images)
        if sample_videos:
            video_inputs.extend(sample_videos)

    processor_kwargs: dict[str, Any] = {
        "text": texts,
        "images": image_inputs or None,
        "videos": video_inputs or None,
        "return_tensors": "pt",
        "padding": True,
    }
    if backend == "qwen3_vl":
        # qwen-vl-utils already resized the image for Qwen3-VL.
        processor_kwargs["do_resize"] = False

    inputs = processor(**processor_kwargs)
    inputs = _move_batch_to_device(inputs, ctx["input_device"])

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=int(spec.get("max_new_tokens", 80)),
            do_sample=False,
            use_cache=True,
        )

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(inputs["input_ids"], generated_ids)
    ]
    return [
        text.strip()
        for text in processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
    ]


def _generate_blip2_batch(
    image_paths: list[str],
    ctx: dict[str, Any],
) -> list[str]:
    """Generate captions with BLIP-2 OPT."""
    import torch
    from PIL import Image

    model = ctx["model"]
    processor = ctx["processor"]
    spec = ctx["spec"]

    images = []
    try:
        images = [Image.open(path).convert("RGB") for path in image_paths]
        inputs = processor(
            images=images,
            text=[str(spec["prompt"])] * len(images),
            return_tensors="pt",
            padding=True,
        )
        target_dtype = ctx["dtype"] if ctx["device"] == "cuda" else torch.float32
        inputs = _move_batch_to_device(
            inputs,
            ctx["input_device"],
            dtype=target_dtype,
        )

        with torch.inference_mode():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=int(spec.get("max_new_tokens", 80)),
                min_new_tokens=int(spec.get("min_new_tokens", 0)),
                num_beams=int(spec.get("num_beams", 1)),
                repetition_penalty=float(spec.get("repetition_penalty", 1.0)),
                length_penalty=float(spec.get("length_penalty", 1.0)),
                use_cache=True,
            )

        return [
            text.strip()
            for text in processor.batch_decode(
                generated_ids,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
        ]
    finally:
        for image in images:
            image.close()


def generate_caption_batch(
    image_paths: list[str | Path],
    ctx: dict[str, Any],
) -> list[str]:
    """Caption a list of images using the backend registered in ctx."""
    image_paths = [str(path) for path in image_paths]
    if not image_paths:
        return []

    backend = ctx["backend"]
    batch_size = int(ctx["spec"].get("batch_size", 1))
    captions: list[str] = []

    for path_batch in _chunked(image_paths, batch_size):
        if backend in {"qwen25_vl", "qwen3_vl"}:
            captions.extend(_generate_qwen_batch(path_batch, ctx))
        elif backend == "blip2":
            captions.extend(_generate_blip2_batch(path_batch, ctx))
        else:
            raise ValueError(f"No inference implementation for backend={backend!r}")

    if len(captions) != len(image_paths):
        raise RuntimeError(
            f"Caption count mismatch: got {len(captions)} outputs "
            f"for {len(image_paths)} images."
        )
    return captions


def extract_task_batch(
    records: list[dict[str, Any]],
    ctx: dict[str, Any],
    config: Any,
    layout: RunLayout,
) -> list[dict[str, Any]]:
    """Create caption annotations for one downloaded frame batch."""
    captions = generate_caption_batch(
        [record["local_image_path"] for record in records],
        ctx,
    )

    outputs: list[dict[str, Any]] = []
    for record, caption in zip(records, captions):
        item = base_annotation(
            record,
            config,
            kind="captioning",
            run_id=layout.run_id,
        )
        item.update({
            "caption": caption,
            "text_value": caption or None,
            "json_value": {
                "caption": caption,
                "model_key": ctx["model_key"],
                "model_id": ctx["model_id"],
                "backend": ctx["backend"],
                "quantization": ctx["spec"].get("quantization", "none"),
            },
        })
        outputs.append(item)
    return outputs


def _select_benchmark_records(
    records: list[dict[str, Any]],
    video_limit: int,
    frames_per_video: int,
) -> list[dict[str, Any]]:
    """Select representative records from distinct videos."""
    by_video: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        by_video.setdefault(str(record["video_id"]), []).append(record)

    selected: list[dict[str, Any]] = []
    for video_id in sorted(by_video)[: max(1, int(video_limit))]:
        candidates = sorted(
            by_video[video_id],
            key=lambda row: (row["frame_idx"], row["keyframe_id"]),
        )
        count = min(max(1, int(frames_per_video)), len(candidates))
        if count == 1:
            chosen = [candidates[len(candidates) // 2]]
        else:
            # Evenly spread representative frames across the video's keyframes.
            indexes = np.linspace(0, len(candidates) - 1, num=count, dtype=int)
            chosen = [candidates[index] for index in indexes]
        selected.extend(chosen)
    return selected


def benchmark_caption_models(
    config: Any,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Benchmark registry models sequentially on representative frames."""
    import torch

    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    batches = [
        str(batch).upper()
        for batch in cfg_value(config, "BENCHMARK_BATCHES", ["L21"])
    ]
    all_records = discover_frame_records(config, bucket, batches, max_frames=None)
    selected = _select_benchmark_records(
        all_records,
        int(cfg_value(config, "BENCHMARK_VIDEO_LIMIT", 5)),
        int(cfg_value(config, "BENCHMARK_FRAMES_PER_VIDEO", 1)),
    )
    unique_videos = sorted({record["video_id"] for record in selected})
    requested_video_count = int(cfg_value(config, "BENCHMARK_VIDEO_LIMIT", 5))
    if len(unique_videos) < requested_video_count:
        raise RuntimeError(
            f"Only found {len(unique_videos)} distinct videos in batches={batches}; "
            f"requested {requested_video_count}."
        )

    layout = make_run_layout(config, "benchmark", "benchmark")
    logger = setup_logging(layout)
    selected = download_frames(selected, layout, client, config)
    logger.info(
        "Benchmark frames=%d videos=%s models=%s",
        len(selected),
        unique_videos,
        cfg_value(config, "BENCHMARK_MODEL_KEYS", []),
    )

    rows: list[dict[str, Any]] = []
    summary_rows: list[dict[str, Any]] = []

    for model_key in cfg_value(config, "BENCHMARK_MODEL_KEYS", []):
        ctx = None
        load_started = time.perf_counter()
        try:
            reset_cuda_peak_memory()
            ctx = load_caption_model(config, logger, str(model_key))
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            load_seconds = time.perf_counter() - load_started
            loaded_memory = cuda_memory_metrics()

            inference_started = time.perf_counter()
            captions = generate_caption_batch(
                [record["local_image_path"] for record in selected],
                ctx,
            )
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            inference_seconds = time.perf_counter() - inference_started
            peak_memory = cuda_memory_metrics()
            frames_count = len(selected)
            seconds_per_frame = inference_seconds / frames_count if frames_count else 0.0

            for record, caption in zip(selected, captions):
                rows.append({
                    "batch_id": record["batch_id"],
                    "video_id": record["video_id"],
                    "keyframe_id": record["keyframe_id"],
                    "frame_idx": record["frame_idx"],
                    "frame_sec": record["frame_sec"],
                    "image_gcs_uri": record["image_gcs_uri"],
                    "model_key": ctx["model_key"],
                    "model_id": ctx["model_id"],
                    "backend": ctx["backend"],
                    "quantization": ctx["spec"].get("quantization", "none"),
                    "caption": caption,
                    "model_load_seconds": round(load_seconds, 4),
                    "inference_seconds_total": round(inference_seconds, 4),
                    "seconds_per_frame": round(seconds_per_frame, 4),
                    "total_seconds": round(load_seconds + inference_seconds, 4),
                    "model_loaded_vram_gb": loaded_memory["vram_allocated_gb"],
                    "peak_vram_allocated_gb": peak_memory["peak_vram_allocated_gb"],
                    "peak_vram_reserved_gb": peak_memory["peak_vram_reserved_gb"],
                })

            summary_rows.append({
                "model_key": ctx["model_key"],
                "model_id": ctx["model_id"],
                "backend": ctx["backend"],
                "quantization": ctx["spec"].get("quantization", "none"),
                "videos": len(unique_videos),
                "frames": frames_count,
                "model_load_seconds": round(load_seconds, 4),
                "inference_seconds_total": round(inference_seconds, 4),
                "seconds_per_frame": round(seconds_per_frame, 4),
                "total_seconds": round(load_seconds + inference_seconds, 4),
                "model_loaded_vram_gb": loaded_memory["vram_allocated_gb"],
                "peak_vram_allocated_gb": peak_memory["peak_vram_allocated_gb"],
                "peak_vram_reserved_gb": peak_memory["peak_vram_reserved_gb"],
            })
        except Exception as exc:
            logger.exception("Benchmark failed for model_key=%s: %s", model_key, exc)
            summary_rows.append({
                "model_key": str(model_key),
                "model_id": get_model_spec(config, str(model_key)).get("hf_id", ""),
                "backend": get_model_spec(config, str(model_key)).get("backend", ""),
                "quantization": get_model_spec(config, str(model_key)).get("quantization", "none"),
                "videos": len(unique_videos),
                "frames": len(selected),
                "error": str(exc),
            })
            if cfg_value(config, "FAIL_FAST", False):
                raise
        finally:
            unload_caption_model(ctx)

    long_df = pd.DataFrame(rows)
    summary_df = pd.DataFrame(summary_rows)

    index_columns = [
        "batch_id",
        "video_id",
        "keyframe_id",
        "frame_idx",
        "frame_sec",
        "image_gcs_uri",
    ]
    value_columns = [
        "caption",
        "seconds_per_frame",
        "model_load_seconds",
        "inference_seconds_total",
        "peak_vram_allocated_gb",
        "peak_vram_reserved_gb",
        "quantization",
    ]
    if long_df.empty:
        comparison_df = pd.DataFrame()
    else:
        comparison_df = long_df.pivot_table(
            index=index_columns,
            columns="model_key",
            values=value_columns,
            aggfunc="first",
        ).reset_index()
        comparison_df.columns = [
            "__".join([str(part) for part in column if str(part)])
            if isinstance(column, tuple)
            else str(column)
            for column in comparison_df.columns
        ]

    if bool(cfg_value(config, "BENCHMARK_SAVE_CSV", True)):
        long_path = layout.artifacts_dir / "benchmark_long.csv"
        comparison_path = layout.artifacts_dir / "benchmark_comparison.csv"
        summary_path = layout.artifacts_dir / "benchmark_model_summary.csv"
        long_df.to_csv(long_path, index=False)
        comparison_df.to_csv(comparison_path, index=False)
        summary_df.to_csv(summary_path, index=False)
        logger.info("Saved benchmark CSV files under %s", layout.artifacts_dir)

    return long_df, comparison_df, summary_df


def run_frame_extractor(
    config: Any,
    batches: list[str],
    max_frames: int | None,
    run_kind: str,
) -> dict[str, Any]:
    """Run captioning with ACTIVE_MODEL_KEY and upload standard artifacts."""
    import torch

    client = make_storage_client(config)
    bucket = client.bucket(resolve_bucket_name(config, require=True))
    layout = make_run_layout(
        config,
        batches[0] if len(batches) == 1 else "all",
        run_kind,
    )
    logger = setup_logging(layout)
    started = time.perf_counter()

    records = discover_frame_records(config, bucket, batches, max_frames=max_frames)
    discovered_count = len(records)
    logger.info("Discovered %d frames for batches=%s", discovered_count, batches)

    processed_keys = load_processed_keyframes(config, bucket, batches)
    if processed_keys:
        before_count = len(records)
        records = [
            record
            for record in records
            if record["keyframe_id"] not in processed_keys
        ]
        skipped = before_count - len(records)
        logger.info("Resume filter skipped %d already processed frames", skipped)
    else:
        skipped = 0

    remaining_count = len(records)
    records = download_frames(records, layout, client, config) if records else []

    processed = 0
    failed = 0
    model_ctx = None
    active_model_key = str(cfg_value(config, "ACTIVE_MODEL_KEY"))
    active_spec = get_model_spec(config, active_model_key)
    model_load_seconds = 0.0

    reset_cuda_peak_memory()
    memory = cuda_memory_metrics()

    try:
        if records:
            load_started = time.perf_counter()
            model_ctx = load_caption_model(config, logger, active_model_key)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            model_load_seconds = time.perf_counter() - load_started

        batch_size = int(cfg_value(config, "PIPELINE_BATCH_SIZE", 8))
        batches_to_run = list(iter_batches(records, batch_size))
        for batch_index, record_batch in enumerate(
            tqdm(
                batches_to_run,
                desc=f"{cfg_value(config, 'EXTRACTOR_NAME')} batches",
                disable=not cfg_value(config, "USE_TQDM", True),
            ),
            start=1,
        ):
            batch_started = time.perf_counter()
            try:
                output_records = extract_task_batch(
                    record_batch,
                    model_ctx,
                    config,
                    layout,
                )
                append_jsonl(layout.annotations_path, output_records)
                processed += len(output_records)
                elapsed = time.perf_counter() - batch_started
                memory = cuda_memory_metrics()
                append_metric(layout.metrics_path, {
                    "run_id": layout.run_id,
                    "batch_index": batch_index,
                    "model_key": active_model_key,
                    "model_id": active_spec["hf_id"],
                    "frames": len(record_batch),
                    "processed": len(output_records),
                    "failed": 0,
                    "seconds": round(elapsed, 3),
                    "frames_per_second": round(len(record_batch) / elapsed, 4) if elapsed else 0,
                    "peak_vram_allocated_gb": memory["peak_vram_allocated_gb"],
                    "peak_vram_reserved_gb": memory["peak_vram_reserved_gb"],
                })
            except Exception as exc:
                failed += len(record_batch)
                append_jsonl(
                    layout.errors_path,
                    [
                        {
                            **base_annotation(
                                record,
                                config,
                                "captioning",
                                layout.run_id,
                            ),
                            "model_key": active_model_key,
                            "model_id": active_spec["hf_id"],
                            "error": str(exc),
                        }
                        for record in record_batch
                    ],
                )
                append_metric(layout.metrics_path, {
                    "run_id": layout.run_id,
                    "batch_index": batch_index,
                    "model_key": active_model_key,
                    "model_id": active_spec["hf_id"],
                    "frames": len(record_batch),
                    "processed": 0,
                    "failed": len(record_batch),
                    "seconds": round(time.perf_counter() - batch_started, 3),
                    "frames_per_second": 0,
                    **cuda_memory_metrics(),
                })
                logger.exception("Batch %d failed: %s", batch_index, exc)
                if "out of memory" in str(exc).lower() and torch.cuda.is_available():
                    torch.cuda.empty_cache()
                if cfg_value(config, "FAIL_FAST", False):
                    raise

            total_seen = processed + failed + skipped
            if (
                total_seen
                and total_seen % int(cfg_value(config, "LOG_EVERY_N_FRAMES", 32))
                < batch_size
            ):
                logger.info(
                    "Progress: %d/%d processed=%d skipped=%d failed=%d remaining=%d",
                    total_seen,
                    discovered_count,
                    processed,
                    skipped,
                    failed,
                    remaining_count,
                )
    finally:
        memory = cuda_memory_metrics()
        unload_caption_model(model_ctx)

    success = failed == 0
    summary = {
        "run_id": layout.run_id,
        "status": "SUCCESS" if success else "COMPLETED_WITH_ERRORS",
        "extractor": cfg_value(config, "EXTRACTOR_NAME"),
        "extractor_version": cfg_value(config, "EXTRACTOR_VERSION"),
        "annotation_version": cfg_value(config, "ANNOTATION_VERSION"),
        "dataset_id": cfg_value(config, "DATASET_ID"),
        "batches": batches,
        "model_key": active_model_key,
        "model_id": active_spec["hf_id"],
        "backend": active_spec["backend"],
        "quantization": active_spec.get("quantization", "none"),
        "planned_frames": discovered_count,
        "processed_frames": processed,
        "skipped_frames": skipped,
        "failed_frames": failed,
        "model_load_seconds": round(model_load_seconds, 3),
        "duration_seconds": round(time.perf_counter() - started, 3),
        "peak_vram_allocated_gb": memory["peak_vram_allocated_gb"],
        "peak_vram_reserved_gb": memory["peak_vram_reserved_gb"],
        "output_prefix": f"gs://{bucket.name}/{layout.output_prefix}",
        "created_at": utc_now(),
    }
    write_json(layout.summary_path, summary)

    if (
        cfg_value(config, "UPLOAD_TO_GCS", True)
        and cfg_value(config, "UPLOAD_RUN_ARTIFACTS", True)
    ):
        upload_standard_artifacts(bucket, layout, success)

    if cfg_value(config, "CLEANUP_LOCAL_FRAMES_AFTER_RUN", True):
        shutil.rmtree(layout.frames_dir, ignore_errors=True)

    return summary


def run_demo(config: Any) -> dict[str, Any]:
    """Run ACTIVE_MODEL_KEY on a small end-to-end sample."""
    return run_frame_extractor(
        config,
        [
            str(batch).upper()
            for batch in cfg_value(config, "DEMO_BATCHES", ["L21"])
        ],
        cfg_value(config, "DEMO_MAX_FRAMES", 32),
        "demo",
    )


def run_full(config: Any) -> list[dict[str, Any]]:
    """Run ACTIVE_MODEL_KEY over all configured batches with a safety guard."""
    if cfg_value(config, "CONFIRM_FULL_RUN", "") != "RUN_FULL_DATASET":
        print(
            'Skipped full run. Set CONFIRM_FULL_RUN = "RUN_FULL_DATASET" '
            "in Parameters and rerun that cell."
        )
        return []

    return [
        run_frame_extractor(
            config,
            [str(batch).upper()],
            cfg_value(config, "FULL_MAX_FRAMES", None),
            f"full_{str(batch).lower()}",
        )
        for batch in cfg_value(config, "BATCHES", [])
    ]


## 4. Dry Run

**Note:** Cell này chỉ kiểm tra credential, manifest discovery và planned GCS paths; không download frame, không load model, không upload output.


In [23]:
dry_summary = dry_run(cfg, max_frames=cfg.DRY_RUN_MAX_FRAMES)
dry_summary


{'status': 'DRY_RUN_OK',
 'bucket': 'aic_ai_2026',
 'batches': ['L21'],
 'planned_frames': 20,
 'sample_records': [{'dataset_id': 'ai_challenge_2025',
   'batch_id': 'L21',
   'video_id': 'L21_V001',
   'video_name': 'L21_V001.mp4',
   'shot_id': 'L21_V001_S0000',
   'shot_start_frame': 0,
   'shot_end_frame': 53,
   'frame_type': 'first',
   'frame_idx': 0,
   'frame_sec': 0.0,
   'timestamp_ms': 0,
   'keyframe_id': 'L21_V001_F000000',
   'image_rel_path': 'L21_V001/shot_0000_first_f000000.jpg',
   'image_gcs_uri': 'gs://aic_ai_2026/processed/keyframes/dataset=ai_challenge_2025/batch=L21/profile=autoshot_v1/video_id=L21_V001/shot_0000_first_f000000.jpg',
   'image_storage_key': 'processed/keyframes/dataset=ai_challenge_2025/batch=L21/profile=autoshot_v1/video_id=L21_V001/shot_0000_first_f000000.jpg',
   'fps': 30.0,
   'profile_version': 'autoshot_v1'},
  {'dataset_id': 'ai_challenge_2025',
   'batch_id': 'L21',
   'video_id': 'L21_V001',
   'video_name': 'L21_V001.mp4',
   'shot_id'

In [24]:
# Benchmark 5 videos: load each model sequentially, compare captions, time, and VRAM.
benchmark_long_df, benchmark_comparison_df, benchmark_summary_df = benchmark_caption_models(cfg)

print("Model-level benchmark summary")
display(benchmark_summary_df)

print("Wide comparison table: one row per test keyframe")
display(benchmark_comparison_df)

print("Long-form raw benchmark records")
display(benchmark_long_df)


RuntimeError: Only found 2 distinct videos in batches=['L21']; requested 5.

## 5. Demo Run

**Note:** Cell này chạy end-to-end trên một mẫu nhỏ bằng `ACTIVE_MODEL_KEY` và upload artifact nếu `UPLOAD_TO_GCS=True`.


In [ ]:
demo_summary = run_demo(cfg)
demo_summary


## 6. Full Run

**Note:** Cell này được khóa an toàn. Chỉ chạy toàn bộ khi đặt `CONFIRM_FULL_RUN = "RUN_FULL_DATASET"` trong Parameters rồi chạy lại cell Parameters.


In [ ]:
full_summaries = run_full(cfg)
full_summaries


## 7. Inspect Latest Local Artifacts

**Note:** Liệt kê các artifact gần nhất, bao gồm JSONL/metrics của extractor và CSV benchmark nếu đã chạy.


In [ ]:
run_root = Path(cfg.RUN_ROOT)
latest = sorted(
    [path for path in run_root.glob("*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)[:5]

if not latest:
    print(f"No runs found under {run_root}")

artifact_names = [
    "artifacts/summary.json",
    "artifacts/annotations.jsonl",
    "artifacts/errors.jsonl",
    "artifacts/metrics.csv",
    "artifacts/benchmark_long.csv",
    "artifacts/benchmark_comparison.csv",
    "artifacts/benchmark_model_summary.csv",
    "run.log",
]

for path in latest:
    print(path)
    for artifact in artifact_names:
        candidate = path / artifact
        if candidate.exists():
            print("  ", candidate, candidate.stat().st_size, "bytes")
